In [ ]:
!pip3 install inference roboflow opencv-python
%pip install "stable-baselines3[extra]"
%pip install supervision
!pip3 install pillow

In [ ]:
import os
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CoreMLExecutionProvider,CPUExecutionProvider]"
os.environ["INFERENCE_LOG_LEVEL"] = "DEBUG"

import onnxruntime as ort
print("available:", ort.get_available_providers())

In [ ]:
import os

roboflow_api_key = os.getenv("API_KEY")

In [ ]:
from inference import get_model

model = get_model(model_id = "akshat-tewari-lz8sp/my-first-project-i6ckj-2-yolo11s-t1", api_key=roboflow_api_key)

In [ ]:
from pathlib import Path

# Extract just the name of each file in the directory
training_dir = Path('./images/training_images')
testing_dir = Path('./images/test_images')

training_files = [file.name for file in training_dir.iterdir() if file.is_file()]
testing_files = [file.name for file in testing_dir.iterdir() if file.is_file()]

filenames = training_files + testing_files
filenames

In [ ]:
import random
from PIL import Image

rand_file = random.choice(testing_files)
rand_file = "./images/test_images/" + rand_file
img = Image.open(rand_file)
img

In [ ]:
import torch

if hasattr(torch, "mps") and not hasattr(torch.mps, "current_device"):
    torch.mps.current_device = lambda: 0

# Your existing setup
device = torch.device("mps")

In [ ]:
import supervision as sv

results = model.infer(img)[0]

detections = sv.Detections.from_inference(results)

annotated_image = sv.BoxAnnotator().annotate(scene=img, detections=detections)
annotated_image = sv.LabelAnnotator().annotate(scene=annotated_image, detections=detections)

sv.plot_image(annotated_image)

In [ ]:
%pip install pyautogui pyobjc-framework-Quartz

In [ ]:
import mss, numpy as np, time
from modules.frame import get_window_region

region = get_window_region("BlueStacks")

CROP_TOP_FRAC = 0.096     # trim top 9.6% — BlueStacks title + menu bar
CROP_RIGHT_FRAC = 0.058   # trim right 5.8% — BlueStacks side toolbar

def grab_cropped(sct, region):
    raw = np.array(sct.grab(region))          # BGRA
    rgb = raw[:, :, :3][:, :, ::-1]           # -> RGB
    h, w = rgb.shape[:2]
    top = int(h * CROP_TOP_FRAC)
    return rgb[top:, :, :]               # trim top and right chrome

cap_times = []
with mss.MSS() as sct:
    for _ in range(3):
        grab_cropped(sct, region)
    for i in range(15):
        t0 = time.perf_counter()
        frame = grab_cropped(sct, region)
        cap_times.append((time.perf_counter() - t0) * 1000)
        print(f"frame {i:2d}  capture={cap_times[-1]:6.1f} ms  shape={frame.shape}")

xs = cap_times[1:]
print(f"\ncapture avg={sum(xs)/len(xs):.1f} ms  (~{1000/(sum(xs)/len(xs)):.0f} FPS)")
Image.fromarray(frame)

In [ ]:
import mss, numpy as np, time, os
from datetime import datetime
from PIL import Image
from modules.frame import get_window_region

region = get_window_region("BlueStacks")
print(region)

out_dir = "images/live_images"
os.makedirs(out_dir, exist_ok=True)

with mss.MSS() as sct:
    for _ in range(3):
        np.array(sct.grab(region))          # warmup

    for i in range(15):
        raw = np.array(sct.grab(region))    # BGRA
        rgb = raw[:, :, :3][:, :, ::-1]     # -> RGB, drop alpha

        ts = datetime.now().strftime("%H-%M-%S-%f")[:-3]  # ms precision, unique
        path = os.path.join(out_dir, f"frame_{i:02d}_{ts}.png")
        Image.fromarray(rgb).save(path)
        print(f"saved {path}")

        time.sleep(0.05)   # small gap so the 15 frames aren't near-identical

In [ ]:
import mss, numpy as np, time
from PIL import Image
from inference import get_model
from modules.frame import get_window_region
from modules.inference import run_inference


CROP_TOP_FRAC = 0.096   # trim top 9.6% — BlueStacks title + menu bar


def crop_chrome(rgb):
    """Trim the BlueStacks top chrome. Same crop everywhere downstream."""
    h = rgb.shape[0]
    top = int(h * CROP_TOP_FRAC)
    return rgb[top:, :, :]


def grab_frame(sct, region):
    """Capture the region and return a cropped RGB frame."""
    raw = np.array(sct.grab(region))        # BGRA
    rgb = raw[:, :, :3][:, :, ::-1]         # -> RGB
    return crop_chrome(rgb)


def warmup(model, region, n=5):
    """First infers compile the graph and are misleadingly slow — burn them here."""
    h = region["height"] - int(region["height"] * CROP_TOP_FRAC)
    dummy = Image.fromarray(np.zeros((h, region["width"], 3), dtype=np.uint8))
    for _ in range(n):
        model.infer(dummy)


def process_frame(sct, region, model, debug=False):
    """Grab one frame, run inference, return (detections, capture_ms, infer_ms)."""
    t0 = time.perf_counter()
    frame = grab_frame(sct, region)
    img = Image.fromarray(frame)
    t1 = time.perf_counter()

    detections = run_inference(img, model, debug)
    t2 = time.perf_counter()

    return detections, (t1 - t0) * 1000, (t2 - t1) * 1000


def stats(name, xs):
    xs = xs[1:]  # drop first (warmup-tainted)
    avg = sum(xs) / len(xs)
    print(f"{name:8s} avg={avg:6.1f} ms  min={min(xs):.1f}  max={max(xs):.1f}")


def benchmark(model, region, frames=15, debug=False):
    cap_times, inf_times = [], []
    with mss.MSS() as sct:
        for _ in range(3):
            grab_frame(sct, region)          # capture warmup

        for i in range(frames):
            detections, cap, inf = process_frame(sct, region, model, debug)
            cap_times.append(cap)
            inf_times.append(inf)
            print(f"frame {i:2d}  capture={cap:6.1f} ms  infer={inf:6.1f} ms  "
                  f"total={cap+inf:6.1f} ms  detections={len(detections)}")

    print()
    stats("capture", cap_times)
    stats("infer", inf_times)
    tot = sum(c + i for c, i in zip(cap_times[1:], inf_times[1:])) / (frames - 1)
    print(f"\nend-to-end: {tot:.1f} ms  (~{1000/tot:.1f} FPS)")


region = get_window_region("BlueStacks")
print(region)

warmup(model, region)
benchmark(model, region, frames=15, debug=True)

In [ ]:
def find_ref_box(detections, class_names):
    """Return ref's (cx, cy, width_px), or None if ref isn't detected."""
    for i, class_id in enumerate(detections.class_id):
        if class_names[class_id] == "ref":
            x1, y1, x2, y2 = detections.xyxy[i]
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
            width_px = abs(x2 - x1)
            return (cx, cy, width_px)
    return None

def find_class_center(detections, class_names, target):
    """Return (cx, cy) of the first detection matching target, or None."""
    for i, class_id in enumerate(detections.class_id):
        if class_names[class_id] == target:
            x1, y1, x2, y2 = detections.xyxy[i]
            return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)
    return None

def _names(detections):
    """Class-name array straight from the detections object."""
    return detections.data["class_name"]


def find_class_center(detections, target):
    """Return (cx, cy) of the first detection whose class_name == target, or None."""
    names = _names(detections)
    for i, name in enumerate(names):
        if name == target:
            x1, y1, x2, y2 = detections.xyxy[i]
            return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)
    return None


def absolute_coords(detections, units_per_span=10.0,
                    keep=("ball", "full_tank", "car", "boost", "nametag")):
    ref = find_class_center(detections, "ref")
    ref2 = find_class_center(detections, "ref2")
    if ref is None or ref2 is None:
        return None
    rx, ry = ref
    span = ((ref2[0]-rx)**2 + (ref2[1]-ry)**2) ** 0.5
    if span == 0:
        return None
    upp = units_per_span / span

    names = _names(detections)
    out = {}
    for i, name in enumerate(names):
        if name not in keep:
            continue
        x1, y1, x2, y2 = detections.xyxy[i]
        cx, cy = (x1+x2)/2.0, (y1+y2)/2.0
        ux = (cx - rx) * upp
        uy = (ry - cy) * upp
        # store units AND pixel center
        out.setdefault(name, []).append((ux, uy, cx, cy))
    return out

def make_to_units(ref_center, ref_width, units_per_ref_width=5.0):
    """
    Build a pixel->unit converter with ref as origin, scaled by ref's width.
    One 'ref width' of pixels == `units_per_ref_width` units, on both axes (square).
    So the scale rides with the camera: zoom-invariant.
    +y up.
    """
    rx, ry = ref_center
    upp = units_per_ref_width / ref_width       # units per pixel, from ref's size

    def to_units(px, py):
        return ((px - rx) * upp,                # right = +x
                (ry - py) * upp)                # up = +y (image y flipped)
    return to_units


def detections_to_units(detections, frame_shape, class_names,
                        units_per_ref_width=5.0):
    """
    Map every detection center into ref-origin, ref-width-scaled unit coordinates.
    Returns dict class_name -> list of (ux, uy), or None if ref not visible.
    frame_shape kept in the signature for interface consistency (unused now).
    """
    ref = find_ref_box(detections, class_names)
    if ref is None:
        return None                             # no origin this frame — hold last state upstream

    rx, ry, rw = ref
    to_units = make_to_units((rx, ry), rw, units_per_ref_width)
    out = {}
    for i, class_id in enumerate(detections.class_id):
        name = class_names[class_id]
        x1, y1, x2, y2 = detections.xyxy[i]
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        out.setdefault(name, []).append(to_units(cx, cy))
    return out

In [ ]:
import os
import cv2
import numpy as np
from datetime import datetime


def draw_coord_system(frame, detections,
                      keep=("ball", "full_tank", "car", "boost", "nametag"),
                      units_per_span=5.0, save=True):
    """
    Overlay a ref-origin coordinate grid + detection boxes and save it.
    Origin = ref center. Scale = ref->ref2 pixel span (cancels camera offset).
    Square units, +y up. Returns annotated BGR image, or None if a marker is missing.
    """
    ref = find_class_center(detections, "ref")
    ref2 = find_class_center(detections, "ref2")
    if ref is None or ref2 is None:
        print("ref or ref2 not detected — no grid drawn")
        return None

    h, w = frame.shape[:2]
    rx, ry = ref
    span = ((ref2[0] - rx) ** 2 + (ref2[1] - ry) ** 2) ** 0.5
    if span == 0:
        print("ref and ref2 coincide — no scale")
        return None

    ppu = span / units_per_span                  # pixels per unit
    upp = units_per_span / span                  # units per pixel

    img = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR).copy()

    grid_color = (90, 90, 90)
    axis_color = (0, 165, 255)      # orange axes (BGR)
    label_color = (255, 255, 255)
    box_color = (0, 220, 0)         # green boxes
    center_color = (0, 0, 255)      # red centers/labels
    ref_color = (255, 200, 0)       # cyan-ish for the markers

    # --- grid: vertical lines at each integer x-unit ---
    max_ux = int(np.ceil((w - rx) * upp))
    min_ux = int(np.floor((0 - rx) * upp))
    for ux in range(min_ux, max_ux + 1):
        px = int(rx + ux * ppu)
        if 0 <= px < w:
            color = axis_color if ux == 0 else grid_color
            thick = 2 if ux == 0 else 1
            cv2.line(img, (px, 0), (px, h), color, thick)
            if ux != 0:
                cv2.putText(img, f"{ux:+d}", (px + 2, int(ry) - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35, label_color, 1)

    # --- grid: horizontal lines at each integer y-unit (+y up) ---
    max_uy = int(np.ceil(ry * upp))
    min_uy = int(np.floor((ry - h) * upp))
    for uy in range(min_uy, max_uy + 1):
        py = int(ry - uy * ppu)
        if 0 <= py < h:
            color = axis_color if uy == 0 else grid_color
            thick = 2 if uy == 0 else 1
            cv2.line(img, (0, py), (w, py), color, thick)
            if uy != 0:
                cv2.putText(img, f"{uy:+d}", (int(rx) + 4, py - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35, label_color, 1)

    # --- origin + both markers ---
    cv2.circle(img, (int(rx), int(ry)), 4, axis_color, -1)
    cv2.putText(img, "(0,0) ref", (int(rx) + 6, int(ry) + 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, axis_color, 1)
    cv2.circle(img, (int(ref2[0]), int(ref2[1])), 4, ref_color, -1)
    cv2.putText(img, "ref2", (int(ref2[0]) + 6, int(ref2[1]) + 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, ref_color, 1)

    # --- detections: box + center + unit coordinate ---
    names = detections.data["class_name"]
    for i, name in enumerate(names):
        # if name not in keep:
        #     continue
        x1, y1, x2, y2 = [int(v) for v in detections.xyxy[i]]
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        ux = (cx - rx) * upp
        uy = (ry - cy) * upp

        cv2.rectangle(img, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(img, name, (x1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, box_color, 1)
        cv2.circle(img, (int(cx), int(cy)), 4, center_color, -1)
        cv2.putText(img, f"({ux:+.1f},{uy:+.1f})", (int(cx) + 6, int(cy)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, center_color, 1)

    if save:
        out_dir = "images/coordinatesystem"
        os.makedirs(out_dir, exist_ok=True)
        ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S-%f")[:-3]
        path = os.path.join(out_dir, f"coords_{ts}.png")
        cv2.imwrite(path, img)
        print(f"saved {path}")

    return img

In [ ]:
import math

def heading_from_velocity(vx, vy, min_speed=0.5, noise_floor=0.5):
    speed = (vx*vx + vy*vy) ** 0.5
    if speed < min_speed:
        return None
    # if a component is below the measured noise floor, treat it as zero
    if abs(vy) < noise_floor and abs(vx) >= noise_floor:
        return 0.0 if vx > 0 else 180.0
    if abs(vx) < noise_floor and abs(vy) >= noise_floor:
        return 90.0 if vy > 0 else -90.0
    return math.degrees(math.atan2(vy, vx))

In [ ]:
class VelocityTracker:
    def __init__(self, alpha=0.5):
        self.prev = None
        self.prev_t = None
        self.vel = {}
        self.alpha = alpha

    def update(self, coords, t=None):
        import time
        if t is None:
            t = time.perf_counter()
        if coords is None:
            return None

        if self.prev is None:
            result = {n: [(ux, uy, 0.0, 0.0, cx, cy) for (ux, uy, cx, cy) in ps]
                      for n, ps in coords.items()}
            self.prev, self.prev_t = coords, t
            self.vel = {n: [(0.0, 0.0)] * len(ps) for n, ps in coords.items()}
            return result

        dt = max(t - self.prev_t, 1e-6)
        result = {}
        for name, positions in coords.items():
            prev_positions = self.prev.get(name, [])
            matched = self._match(positions, prev_positions)
            vels, out = [], []
            for j, (ux, uy, cx, cy) in enumerate(positions):
                pi = matched[j]
                if pi is None:
                    vx = vy = 0.0
                else:
                    pux, puy = prev_positions[pi][0], prev_positions[pi][1]
                    vx, vy = (ux - pux) / dt, (uy - puy) / dt
                    prev_v = self.vel.get(name, [])
                    if pi < len(prev_v):
                        pvx, pvy = prev_v[pi]
                        vx = self.alpha*vx + (1-self.alpha)*pvx
                        vy = self.alpha*vy + (1-self.alpha)*pvy
                vels.append((vx, vy))
                out.append((ux, uy, vx, vy, cx, cy))
            result[name] = out
            self.vel[name] = vels
        self.prev, self.prev_t = coords, t
        return result

    @staticmethod
    def _match(current, previous):
        matched = [None]*len(current); used = set()
        for i, cur in enumerate(current):
            cx, cy = cur[2], cur[3]          # match on pixel center
            best, bd = None, float("inf")
            for j, pv in enumerate(previous):
                if j in used: continue
                d = (cx-pv[2])**2 + (cy-pv[3])**2
                if d < bd: best, bd = j, d
            if best is not None:
                matched[i] = best; used.add(best)
        return matched

In [ ]:
import numpy as np


def _center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def find_my_car_box(detections):
    """
    Pick YOUR car as the 'car' detection closest to the boost indicator
    (full_tank or boost), which renders attached to your car.
    Returns (x1,y1,x2,y2) or None.
    Falls back to the only car if there's just one; None if no cars.
    """
    names = detections.data["class_name"]

    car_boxes = [tuple(int(v) for v in detections.xyxy[i])
                 for i, n in enumerate(names) if n == "car"]
    if not car_boxes:
        return None
    if len(car_boxes) == 1:
        return car_boxes[0]                      # only one car, it's yours

    # boost indicator center(s) — full_tank or boost
    boost_centers = [_center(tuple(detections.xyxy[i]))
                     for i, n in enumerate(names) if n in ("full_tank", "boost")]
    if not boost_centers:
        return car_boxes[0]                      # no indicator this frame — best guess

    # use the first indicator; pick the car nearest to it
    bx, by = boost_centers[0]
    best_box, best_d = None, float("inf")
    for box in car_boxes:
        ccx, ccy = _center(box)
        d = (ccx - bx) ** 2 + (ccy - by) ** 2
        if d < best_d:
            best_d, best_box = d, box
    return best_box

import cv2
import numpy as np
import math


# HSV range for the cyan control arrow (OpenCV: H 0-179, S/V 0-255)
ARROW_LOW  = np.array([98, 40, 140])
ARROW_HIGH = np.array([118, 255, 255])


def car_heading_from_arrow(frame_rgb, car_box, pad=0.6, min_pixels=15):
    """
    Estimate car facing from the cyan control arrow near the car.
    frame_rgb : cropped RGB frame (same one used for detection)
    car_box   : (x1, y1, x2, y2) pixel box of YOUR car
    Returns (angle_deg, (arrow_cx, arrow_cy), mask_crop) or (None, None, None).
    angle: 0°=+x right, 90°=+y up (matches your coord convention).
    """
    x1, y1, x2, y2 = car_box
    bw, bh = x2 - x1, y2 - y1
    H, W = frame_rgb.shape[:2]

    # expand box by `pad` on each side to include the adjacent arrow
    cx1 = max(0, int(x1 - bw * pad)); cy1 = max(0, int(y1 - bh * pad))
    cx2 = min(W, int(x2 + bw * pad)); cy2 = min(H, int(y2 + bh * pad))
    crop = frame_rgb[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None, None, None

    hsv = cv2.cvtColor(cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, ARROW_LOW, ARROW_HIGH)

    # clean speckle
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    ys, xs = np.where(mask > 0)
    if len(xs) < min_pixels:
        return None, None, None

    # arrow centroid in full-frame pixel coords
    acx = cx1 + xs.mean()
    acy = cy1 + ys.mean()

    # car center in full-frame pixel coords
    ccx = (x1 + x2) / 2.0
    ccy = (y1 + y2) / 2.0

    # vector car->arrow. flip y so +y is up (image y grows downward)
    dx = acx - ccx
    dy = -(acy - ccy)
    if dx == 0 and dy == 0:
        return None, None, None

    angle = math.degrees(math.atan2(dy, dx))
    return angle, (acx, acy), mask

In [ ]:
import cv2
import numpy as np
import math


# HSV range for the pink/magenta control arrow (OpenCV H 0-179)
PINK_LOW  = np.array([143, 70, 140])
PINK_HIGH = np.array([176, 255, 255])


def find_pink_arrow(frame_rgb, min_pixels=20):
    """
    Locate the pink control arrow in the full frame.
    Returns (arrow_cx, arrow_cy) in pixel coords, or None.
    Takes the largest connected magenta blob to avoid stray speckle.
    """
    hsv = cv2.cvtColor(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, PINK_LOW, PINK_HIGH)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    # largest component excluding background (label 0)
    areas = stats[1:, cv2.CC_STAT_AREA]
    idx = 1 + int(np.argmax(areas))
    if stats[idx, cv2.CC_STAT_AREA] < min_pixels:
        return None
    cx, cy = centroids[idx]
    return (float(cx), float(cy))


def _center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def car_heading_via_pink(frame_rgb, detections):
    """
    Find your car (nearest the pink arrow) and its heading (car->arrow vector).
    Returns (angle_deg, my_car_box, arrow_center) or (None, None, None).
    angle: 0°=+x right, 90°=+y up (matches your coord convention).
    """
    arrow = find_pink_arrow(frame_rgb)
    if arrow is None:

        return None, None, None

    names = detections.data["class_name"]
    car_boxes = [tuple(int(v) for v in detections.xyxy[i])
                 for i, n in enumerate(names) if n == "car"]
    if not car_boxes:
        return None, None, arrow

    ax, ay = arrow
    # your car = the one nearest the pink arrow
    my_box = min(car_boxes, key=lambda b: (_center(b)[0]-ax)**2 + (_center(b)[1]-ay)**2)

    ccx, ccy = _center(my_box)
    dx = ax - ccx
    dy = -(ay - ccy)                 # flip y so +y is up
    if dx == 0 and dy == 0:
        return None, my_box, arrow

    angle = math.degrees(math.atan2(dy, dx))
    return angle, my_box, arrow

In [ ]:
def identify_my_car(tracked, detections):
    """
    Return the pixel center (cx,cy) of YOUR car = the tracked car nearest
    the boost indicator (full_tank/boost). None if undecidable.
    """
    cars = tracked.get("car", [])
    if not cars:
        return None
    if len(cars) == 1:
        return (cars[0][4], cars[0][5])

    names = _names(detections)
    boosts = [((detections.xyxy[i][0]+detections.xyxy[i][2])/2.0,
               (detections.xyxy[i][1]+detections.xyxy[i][3])/2.0)
              for i, n in enumerate(names) if n in ("full_tank", "boost")]
    if not boosts:
        return (cars[0][4], cars[0][5])          # no indicator — best guess
    bx, by = boosts[0]
    my = min(cars, key=lambda c: (c[4]-bx)**2 + (c[5]-by)**2)
    return (my[4], my[5])

In [ ]:
import cv2
import numpy as np


GREEN_LOW  = np.array([35, 50, 50])
GREEN_HIGH = np.array([90, 255, 255])


import cv2
import numpy as np


GREEN_LOW  = np.array([35, 50, 50])
GREEN_HIGH = np.array([90, 255, 255])


def read_can_jump(frame_rgb, box, full_tank=False):
    """
    True if the jump indicator is green.
    - boost class: jump dot is in the CENTER of the ring.
    - full_tank class: jump circle is on TOP of the triangle.
    Returns True / False, or None if the crop is empty.
    """
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = frame_rgb[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0:
        return None

    h, w = crop.shape[:2]
    hsv = cv2.cvtColor(cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, GREEN_LOW, GREEN_HIGH) > 0

    # sample location depends on indicator type
    cx = w / 2.0
    cy = h * 0.25 if full_tank else h / 2.0    # top for full_tank, center for boost
    rmax = min(w, h) / 2.0

    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    dot = dist < rmax * 0.35

    frac = (green & dot).sum() / max(dot.sum(), 1)
    return bool(frac > 0.5)

In [ ]:
import time
import threading
from pynput import keyboard


class KeyListener:
    """
    Runs a keyboard listener on its own thread, logging timestamped events.
    poll() is called each inference tick (~every 50ms) to classify what
    happened in the window since the last poll.
    """
    def __init__(self, tracked, flip_gap=0.3):
        self.tracked = tracked
        self.flip_gap = flip_gap          # max gap between two taps to be a flip
        self._held = set()
        self._space_events = []           # timestamps of space keydowns
        self._lock = threading.Lock()
        self._last_poll = time.perf_counter()
        self._listener = keyboard.Listener(on_press=self._press, on_release=self._release)

    def _name(self, key):
        try:
            c = key.char
            return c.lower() if c else None
        except AttributeError:
            return key.name

    def _press(self, key):
        n = self._name(key)
        if n in self.tracked:
            with self._lock:
                self._held.add(n)
                if n == "space":
                    self._space_events.append(time.perf_counter())

    def _release(self, key):
        n = self._name(key)
        if n in self.tracked:
            with self._lock:
                self._held.discard(n)

    def start(self):
        self._listener.start()

    def stop(self):
        self._listener.stop()

    def poll(self):
        """
        Classify space activity since last poll into jump / flip.
        Returns dict of current action state.
        """
        with self._lock:
            now = time.perf_counter()
            held = set(self._held)
            # take all space events, sorted
            evs = sorted(self._space_events)
            self._space_events = []
            self._last_poll = now

        jump = False
        flip = False
        i = 0
        while i < len(evs):
            if i + 1 < len(evs) and (evs[i+1] - evs[i]) <= self.flip_gap:
                flip = True                 # two taps close together
                i += 2
            else:
                jump = True                 # lone tap
                i += 1

        return {
            "keys": sorted(held),
            "jump": jump,
            "flip": flip,
            "boosting": "shift" in held,
        }

In [ ]:
import os
import cv2
import numpy as np


def get_boost(frame_rgb, box, save_debug=False, debug_dir="images/annotated"):
    """
    Boost fraction from the ring, reading the RGB frame + box directly.
    Detects green, red, AND bright/white fill (covers all boost-bar states).
    frame_rgb is RGB (from grab_frame); converted to BGR internally to match
    the HSV thresholds. Returns proportion 0.0-1.0 (or None if crop empty).
    """
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = frame_rgb[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0:
        return None

    img = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)      # match BGR HSV pipeline
    h, w, _ = img.shape
    center_x, center_y = w // 2, h // 2

    outer_radius = int(min(w, h) * 0.55)
    inner_radius = int(min(w, h) * 0.42)

    ring_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(ring_mask, (center_x, center_y), outer_radius, 255, -1)
    cv2.circle(ring_mask, (center_x, center_y), inner_radius, 0, -1)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # green fill
    mask_green = cv2.inRange(hsv, np.array([35, 100, 100]), np.array([85, 255, 255]))
    # red fill (wraps hue 0 and 180)
    mask_red = (cv2.inRange(hsv, np.array([0, 120, 120]),   np.array([10, 255, 255])) |
                cv2.inRange(hsv, np.array([170, 120, 120]), np.array([180, 255, 255])))
    # bright/white fill
    mask_white = cv2.inRange(hsv, np.array([0, 0, 200]), np.array([180, 60, 255]))

    active_fill = cv2.bitwise_or(cv2.bitwise_or(mask_green, mask_red), mask_white)

    boost_ring = cv2.bitwise_and(active_fill, active_fill, mask=ring_mask)
    total_ring_pixels = cv2.countNonZero(ring_mask)
    boost_ring_pixels = cv2.countNonZero(boost_ring)
    proportion = (boost_ring_pixels / total_ring_pixels) if total_ring_pixels > 0 else 0.0

    # if save_debug:
    #     annotated = img.copy()
    #     cv2.circle(annotated, (center_x, center_y), outer_radius, (255, 0, 0), 1)
    #     cv2.circle(annotated, (center_x, center_y), inner_radius, (255, 0, 0), 1)
    #     annotated[boost_ring > 0] = [255, 255, 0]
    #     os.makedirs(debug_dir, exist_ok=True)
    #     from random import randint
    #     path = os.path.join(debug_dir, f"boost_annotated_{randint(1,100000)}.png")
    #     cv2.imwrite(path, annotated)

    return proportion

In [ ]:
import time
import threading
from pynput import mouse


class ClickListener:
    """Logs timestamped left-clicks on a thread; poll() classifies jump vs flip."""
    def __init__(self, flip_gap=0.18):
        self.flip_gap = flip_gap          # two clicks within this = flip
        self._clicks = []
        self._lock = threading.Lock()
        self._listener = mouse.Listener(on_click=self._on_click)

    def _on_click(self, x, y, button, pressed):
        if pressed and button == mouse.Button.left:   # button-down only
            with self._lock:
                self._clicks.append(time.perf_counter())

    def start(self):
        self._listener.start()

    def stop(self):
        self._listener.stop()

    def poll(self):
        """
        Classify clicks since last poll into jump / flip.
        Uses a rolling hold: a lone click isn't resolved as a jump until it's
        old enough (flip_gap) that no second click can still pair with it —
        so a flip whose two clicks straddle two polls isn't misread as jumps.
        """
        now = time.perf_counter()
        with self._lock:
            evs = sorted(self._clicks)
            keep = []
            jump = flip = False
            i = 0
            while i < len(evs):
                t = evs[i]
                if i + 1 < len(evs) and (evs[i+1] - t) <= self.flip_gap:
                    flip = True
                    i += 2                    # consume the pair
                elif now - t >= self.flip_gap:
                    jump = True               # old enough, no partner came
                    i += 1
                else:
                    keep.append(t)            # too young, might still pair — hold
                    i += 1
            self._clicks = keep
        return {"jump": jump, "flip": flip}

In [ ]:
# TRACKED_KEYS = {"a", "d", "w", "s", "shift"}

# region = get_window_region("BlueStacks")
# tracker = VelocityTracker(alpha=0.1)
# keys = KeyListener(TRACKED_KEYS)
# clicks = ClickListener()
# keys.start()
# clicks.start()
# prev_frame = {}
# it = 0
# with mss.MSS() as sct:
#     prev_frame["my_car"] = {"heading": 180.0}
#     try:
#         while it < 500:
#             frame = grab_frame(sct, region)
#             detections = run_inference(Image.fromarray(frame), model)
#             coords = absolute_coords(detections)
#             tracked = tracker.update(coords)
#             if not tracked:
#                 continue

#             my_center = identify_my_car(tracked, detections)
#             arrow_heading, _, _ = car_heading_via_pink(frame, detections)

#             # --- can-jump: read center dot if boost/full_tank present, else keep previous ---
#             # --- can-jump + boost fraction from boost/full_tank, else keep previous ---
#             names = detections.data["class_name"]
#             can_jump = None
#             boost_frac = None
#             for i, n in enumerate(names):
#                 if n in ("boost", "full_tank"):
#                     box = detections.xyxy[i]
#                     can_jump = read_can_jump(frame, box, full_tank=(n == "full_tank"))
#                     boost_frac = 1.0 if n == "full_tank" else get_boost(frame, box)
#                     # save the boost crop on frame 100 for inspection
#                     if it == 100:
#                         from random import randint
#                         import os
#                         from PIL import Image as PILImage
#                         x1, y1, x2, y2 = [int(v) for v in box]
#                         crop = frame[max(0, y1):y2, max(0, x1):x2]
#                         out_dir = "images/boost"
#                         os.makedirs(out_dir, exist_ok=True)
#                         path = os.path.join(out_dir, f"boost_it{it}_{n}{randint(1, 10000)}.png")
#                         PILImage.fromarray(crop).save(path)
#                         print(f"saved boost crop -> {path}  (class={n}, "
#                             f"box={x1},{y1},{x2},{y2})")
#                     break
#             if can_jump is None:
#                 can_jump = prev_frame.get("boost_state", {}).get("can_jump")
#             if boost_frac is None:
#                 boost_frac = prev_frame.get("boost_state", {}).get("boost_frac")
#             prev_frame["boost_state"] = {"can_jump": can_jump, "boost_frac": boost_frac}

#             # --- poll the key listener once per inference tick ---
#             key_action = keys.poll()               # keys held, boosting
#             click_action = clicks.poll()           # jump, flip
#             action = {**key_action, **click_action}
#             print(f"it={it} action={action}")

#             for name in ("ball", "full_tank", "car", "boost", "nametag"):
#                 for k, (ux, uy, vx, vy, cx, cy) in enumerate(tracked.get(name, [])):
#                     speed = (vx**2 + vy**2) ** 0.5

#                     if name == "car":
#                         is_mine = (my_center is not None and
#                                    (cx-my_center[0])**2 + (cy-my_center[1])**2 < 40**2)
#                         identity = "my_car" if is_mine else "opp_car"

#                         if is_mine:
#                             heading = arrow_heading
#                             source = "arrow"
#                             if heading is None:
#                                 heading = heading_from_velocity(vx, vy)
#                                 source = "vel"
#                             if heading is None:
#                                 heading = prev_frame.get("my_car", {}).get("heading")
#                                 source = "prev"

#                             hstr = f"{heading:+6.1f}° ({source})" if heading is not None else " --- "
#                             print(f"{identity:8s} pos=({ux:+.2f},{uy:+.2f}) "
#                                   f"vel=({vx:+.2f},{vy:+.2f}) speed={speed:.2f} heading={hstr} "
#                                   f"jump={can_jump} action={action} it={it} boost={boost_frac}")

#                             entry = {"ux": ux, "uy": uy, "vx": vx, "vy": vy, "cx": cx, "cy": cy}
#                             entry["heading"] = heading if heading is not None \
#                                                else prev_frame.get("my_car", {}).get("heading")
#                             prev_frame["my_car"] = entry
#                         else:
#                             print(f"{identity:8s} pos=({ux:+.2f},{uy:+.2f}) "
#                                   f"vel=({vx:+.2f},{vy:+.2f}) speed={speed:.2f} it={it}")
#                             prev_frame["opp_car"] = {"ux": ux, "uy": uy, "vx": vx, "vy": vy,
#                                                      "cx": cx, "cy": cy}
#                     else:
#                         print(f"{name:8s} pos=({ux:+.2f},{uy:+.2f}) "
#                               f"vel=({vx:+.2f},{vy:+.2f}) speed={speed:.2f}")
#                         prev_frame[name] = {"ux": ux, "uy": uy, "vx": vx, "vy": vy,
#                                             "cx": cx, "cy": cy}
#             it += 1
#     finally:
#         keys.stop()

# prev_frame

In [ ]:
import json
import time
import threading
from pynput import keyboard


# --- the keys Sideswipe uses; adjust to your actual bindings ---
TRACKED_KEYS = {"a", "d", "w", "s", "space", "shift", "j", "k", "l"}  # example set


class KeyStateTracker:
    """Tracks which tracked keys are currently held, thread-safe."""
    def __init__(self, tracked=TRACKED_KEYS):
        self.tracked = tracked
        self._held = set()
        self._lock = threading.Lock()
        self._listener = keyboard.Listener(on_press=self._press, on_release=self._release)

    def _key_name(self, key):
        try:
            return key.char           # normal chars
        except AttributeError:
            return key.name           # special keys: space, shift, etc.

    def _press(self, key):
        n = self._key_name(key)
        if n in self.tracked:
            with self._lock:
                self._held.add(n)

    def _release(self, key):
        n = self._key_name(key)
        if n in self.tracked:
            with self._lock:
                self._held.discard(n)

    def start(self):
        self._listener.start()

    def stop(self):
        self._listener.stop()

    def snapshot(self):
        """The set of keys held right now."""
        with self._lock:
            return set(self._held)


def obs_to_vector(tracked, prev_frame):
    """
    Build the observation record from your tracked detections + prev_frame.
    Returns a flat dict — the features BC will train on.
    Missing objects -> None (handle in BC preprocessing).
    """
    def get(name, k=0):
        items = tracked.get(name, [])
        if k < len(items):
            ux, uy, vx, vy, cx, cy = items[k]
            return {"ux": ux, "uy": uy, "vx": vx, "vy": vy}
        return None

    return {
        "ball": get("ball"),
        "my_car": prev_frame.get("my_car"),      # includes heading
        "opp_car": prev_frame.get("opp_car"),
        "full_tank": get("full_tank"),
        "boost": get("boost"),
    }

In [ ]:
import json
import os
import time
import numpy as np

OBS_VERSION = 1
OBJECTS = ["ball", "my_car", "opp_car", "nametag", "boost_icon", "my_goal", "opp_goal"]
TRACKED_KEYS = {"a", "d", "w", "s", "shift"}
OUT_PATH = "data/bc.jsonl"


# ---------- observation + action encoders ----------
def _feat(src):
    if src and src.get("ux") is not None:
        return [1.0, src["ux"], src["uy"], src.get("vx", 0.0), src.get("vy", 0.0)]
    return [0.0, 0.0, 0.0, 0.0, 0.0]


def resolve_goals(tracked):
    my_goal, opp_goal = None, None
    for (ux, uy, vx, vy, cx, cy) in tracked.get("goal", []):
        g = {"ux": ux, "uy": uy, "vx": vx, "vy": vy}
        if ux >= 0:
            my_goal = g
        else:
            opp_goal = g
    return my_goal, opp_goal


def build_observation(tracked, prev_frame, can_jump, boost_amount):
    def from_tracked(name):
        items = tracked.get(name, [])
        if items:
            ux, uy, vx, vy, cx, cy = items[0]
            return {"ux": ux, "uy": uy, "vx": vx, "vy": vy}
        return None

    sources = {
        "ball":       prev_frame.get("ball"),
        "my_car":     prev_frame.get("my_car"),
        "opp_car":    prev_frame.get("opp_car"),
        "nametag":    from_tracked("nametag"),
        "boost_icon": from_tracked("boost") or from_tracked("full_tank"),
        "my_goal":    prev_frame.get("my_goal"),
        "opp_goal":   prev_frame.get("opp_goal"),
    }

    vec = []
    for name in OBJECTS:
        vec += _feat(sources.get(name))

    h = prev_frame.get("my_car", {}).get("heading")
    if h is None:
        vec += [0.0, 0.0, 0.0]
    else:
        r = np.radians(h)
        vec += [1.0, float(np.sin(r)), float(np.cos(r))]

    boost_val = float(boost_amount) if boost_amount is not None else 0.0
    vec += [1.0 if boost_amount is not None else 0.0, boost_val]
    vec += [1.0 if can_jump else 0.0]
    return vec


def build_action(key_action, click_action):
    held = set(key_action.get("keys", []))
    return {
        "left":  1 if "a" in held else 0,
        "right": 1 if "d" in held else 0,
        "up":    1 if "w" in held else 0,
        "down":  1 if "s" in held else 0,
        "boost": 1 if key_action.get("boosting") else 0,
        "jump":  1 if click_action.get("jump") else 0,
        "flip":  1 if click_action.get("flip") else 0,
    }


In [ ]:
import json, os, time
import numpy as np

OUT_PATH = "data/10ksample.jsonl"
TRACKED_KEYS = {"a", "d", "w", "s", "shift"}

region = get_window_region("BlueStacks")
tracker = VelocityTracker(alpha=0.1)
keys = KeyListener(TRACKED_KEYS); keys.start()
clicks = ClickListener(); clicks.start()
prev_frame = {"my_car": {"heading": 180.0}}

os.makedirs("data", exist_ok=True)
f = open(OUT_PATH, "a")
it = 0
jump_count = flip_count = 0

try:
    with mss.MSS() as sct:
        while True:
            frame = grab_frame(sct, region)
            detections = run_inference(Image.fromarray(frame), model)
            coords = absolute_coords(detections)
            tracked = tracker.update(coords)
            if not tracked:
                continue

            names = detections.data["class_name"]
            my_center = identify_my_car(tracked, detections)
            arrow_heading, _, _ = car_heading_via_pink(frame, detections)

            # can-jump + boost fraction
            can_jump, boost_frac = None, None
            for i, n in enumerate(names):
                if n in ("boost", "full_tank"):
                    box = detections.xyxy[i]
                    can_jump = read_can_jump(frame, box, full_tank=(n == "full_tank"))
                    boost_frac = 1.0 if n == "full_tank" else get_boost(frame, box)
                    break
            bs = prev_frame.get("boost_state", {})
            if can_jump is None:  can_jump = bs.get("can_jump")
            if boost_frac is None: boost_frac = bs.get("boost_frac")
            prev_frame["boost_state"] = {"can_jump": can_jump, "boost_frac": boost_frac}

            # goals
            my_goal, opp_goal = resolve_goals(tracked)
            if my_goal:  prev_frame["my_goal"] = my_goal
            if opp_goal: prev_frame["opp_goal"] = opp_goal

            # ball + car identities + heading
            for name in ("ball", "car", "nametag"):
                for k, (ux, uy, vx, vy, cx_, cy_) in enumerate(tracked.get(name, [])):
                    if name == "car":
                        is_mine = (my_center is not None and
                                   (cx_-my_center[0])**2 + (cy_-my_center[1])**2 < 40**2)
                        if is_mine:
                            heading = arrow_heading
                            if heading is None: heading = heading_from_velocity(vx, vy)
                            if heading is None: heading = prev_frame.get("my_car", {}).get("heading")
                            prev_frame["my_car"] = {"ux":ux,"uy":uy,"vx":vx,"vy":vy,
                                                    "cx":cx_,"cy":cy_,"heading":heading}
                        else:
                            prev_frame["opp_car"] = {"ux":ux,"uy":uy,"vx":vx,"vy":vy,"cx":cx_,"cy":cy_}
                    elif name == "ball":
                        prev_frame["ball"] = {"ux":ux,"uy":uy,"vx":vx,"vy":vy,"cx":cx_,"cy":cy_}

            # build + write record
            obs_vec = build_observation(tracked, prev_frame, can_jump, boost_frac)
            action = build_action(keys.poll(), clicks.poll())
            f.write(json.dumps({"v": OBS_VERSION, "t": time.perf_counter(),
                                "obs": obs_vec, "action": action}) + "\n")

            jump_count += action["jump"]; flip_count += action["flip"]
            it += 1
            if it % 100 == 0:
                print(f"logged {it}  (jumps={jump_count} flips={flip_count})")
            time.sleep(.005)
except KeyboardInterrupt:
    print(f"stopped at {it} frames")
finally:
    f.close(); keys.stop(); clicks.stop()
    print(f"saved {it} frames to {OUT_PATH}  jumps={jump_count} flips={flip_count}")

In [ ]:
from torch import nn
class BCPolicy(nn.Module):
    def __init__(self, in_dim, n_actions=7, hidden=128):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        # one logit per action (independent sigmoids = multi-label)
        self.head = nn.Linear(hidden, n_actions)
    def forward(self, x):
        return self.head(self.trunk(x))       # logits
def load_bc(path="models/bc_policy_v1.pt", device=None):
    """Rebuild the model + normalizer from disk. Returns (model, (mean, std))."""
    device = device or DEVICE
    ckpt = torch.load(path, map_location=device, weights_only=False)
    bc_model = BCPolicy(ckpt["in_dim"], ckpt["n_actions"]).to(device)
    bc_model.load_state_dict(ckpt["state_dict"])
    bc_model.eval()
    norm = (ckpt["mean"], ckpt["std"])
    print(f"loaded BC policy <- {path}  (in_dim={ckpt['in_dim']})")
    return bc_model, norm

ACTION_KEYS = ["left", "right", "up", "down", "boost", "jump", "flip"]
RARE = ["jump", "flip"]                      # heads to up-weight
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
import pyautogui
import numpy as np
import torch
import time

pyautogui.PAUSE = 0  # we manage our own timing
pyautogui.FAILSAFE = True  # slam mouse to a corner to kill it

MOVE_KEYS = ["left", "right", "up", "down", "boost"]
KEY_MAP = {"left": "a", "right": "d", "up": "w", "down": "s", "boost": "shift"}
THRESH = {"left": 0.5, "right": 0.5, "up": 0.5, "down": 0.5, "boost": 0.5}


def policy_action(bc_model, norm, obs_vec):
    mean, std = norm
    x = (np.array(obs_vec, dtype=np.float32) - mean) / std
    with torch.no_grad():
        p = (
            torch.sigmoid(bc_model(torch.from_numpy(x).unsqueeze(0).to(DEVICE)))
            .cpu()
            .numpy()[0]
        )
    move = {
        k: int(p[i] > THRESH[k]) for i, k in enumerate(ACTION_KEYS) if k in MOVE_KEYS
    }
    jump = int(p[ACTION_KEYS.index("jump")] > THRESH.get("jump", 0.3))
    flip = int(p[ACTION_KEYS.index("flip")] > THRESH.get("flip", 0.3))
    return move, jump, flip


def apply_action(move, jump, flip, held_state, click_fn):
    """Press/release movement keys to match `move`; click for jump/flip."""
    for act, key in KEY_MAP.items():
        want = bool(move.get(act, 0))
        if want and not held_state[act]:
            pyautogui.keyDown(key)
            held_state[act] = True
        elif not want and held_state[act]:
            pyautogui.keyUp(key)
            held_state[act] = False
    if flip:
        click_fn()
        time.sleep(0.05)
        click_fn() # double jump by clicking
    elif jump:
        click_fn()  # jump = 1 click


def run_bot(model, bc_model, norm, max_it=2500):
    region = get_window_region("BlueStacks")
    tracker = VelocityTracker(alpha=0.1)
    prev_frame = {"my_car": {"heading": 180.0}} # has all properties of car in this dictionary
    held_state = {k: False for k in KEY_MAP}
    # click helper — clicks the jump button (white circle, bottom-right)
    JUMP_BTN_FX = 0.7  # fraction of window width
    JUMP_BTN_FY = 0.85  # fraction of window height
    jbx = region["left"] + int(region["width"] * JUMP_BTN_FX)
    jby = region["top"] + int(region["height"] * JUMP_BTN_FY)

    def click_fn():
        pyautogui.mouseDown(jbx, jby)
        time.sleep(0.03)          # hold so it acc works
        pyautogui.mouseUp(jbx, jby)

    it = 0
    try:
        with mss.MSS() as sct:
            while it < max_it:
                frame = grab_frame(sct, region)
                detections = run_inference(
                    Image.fromarray(frame), model
                )  # YOLO model
                coords = absolute_coords(detections)
                tracked = tracker.update(coords)
                if not tracked:
                    continue

                names = detections.data["class_name"]
                my_center = identify_my_car(tracked, detections)
                arrow_heading, _, _ = car_heading_via_pink(frame, detections)

                can_jump = None
                for i, n in enumerate(names):
                    if n in ("boost", "full_tank"):
                        can_jump = read_can_jump(
                            frame, detections.xyxy[i], full_tank=(n == "full_tank")
                        )
                        break
                if can_jump is None:
                    can_jump = prev_frame.get("boost_state", {}).get("can_jump")
                prev_frame["boost_state"] = {"can_jump": can_jump}

                my_goal, opp_goal = resolve_goals(tracked)
                if my_goal:
                    prev_frame["my_goal"] = my_goal
                if opp_goal:
                    prev_frame["opp_goal"] = opp_goal

                for name in ("ball", "car"):
                    for k, (ux, uy, vx, vy, cx_, cy_) in enumerate(
                        tracked.get(name, [])
                    ):
                        if name == "car":
                            is_mine = (
                                my_center is not None
                                and (cx_ - my_center[0]) ** 2
                                + (cy_ - my_center[1]) ** 2
                                < 40**2
                            )
                            if is_mine:
                                heading = (
                                    arrow_heading
                                    or heading_from_velocity(vx, vy)
                                    or prev_frame.get("my_car", {}).get("heading")
                                )
                                prev_frame["my_car"] = {
                                    "ux": ux, "uy": uy, "vx": vx, "vy": vy,
                                    "cx": cx_, "cy": cy_, "heading": heading,
                                }
                            else:
                                prev_frame["opp_car"] = {
                                    "ux": ux, "uy": uy, "vx": vx, "vy": vy,
                                    "cx": cx_, "cy": cy_,
                                }
                        else:
                            prev_frame["ball"] = {
                                "ux": ux, "uy": uy, "vx": vx, "vy": vy,
                                "cx": cx_, "cy": cy_,
                            }

                obs_vec = build_observation(tracked, prev_frame, can_jump, None)
                move, jump, flip = policy_action(bc_model, norm, obs_vec)
                print(f"move: {move}, jump: {jump}, flip: {flip}")
                apply_action(move, jump, flip, held_state, click_fn)

                it += 1
    except KeyboardInterrupt:
        print("stopped")
    finally:
        # release everything so keys don't stay stuck down
        for act, key in KEY_MAP.items():
            if held_state[act]:
                pyautogui.keyUp(key)
        print("all keys released")


bc_model, norm = load_bc()
# model is yolo detector
# bc_model is game ai
run_bot(model, bc_model, norm)   